### Database-Style Joins

In [1]:
using DataFrames

In [11]:
people = DataFrame(ID=[10, 20, 40], 
                   Name=["Timmy", "John Doe", "Jane Doe"])

,ID,Name
,Int64,String
1,10,Timmy
2,20,John Doe
3,40,Jane Doe


In [12]:
jobs = DataFrame(ID=[20, 40, 60], Job=["Doctor", "Lawyer", "Astronaut"])

,ID,Job
,Int64,String
1,20,Doctor
2,40,Lawyer
3,60,Astronaut


In [13]:
innerjoin(people, jobs, on=:ID)

,ID,Name,Job
,Int64,String,String
1,20,John Doe,Doctor
2,40,Jane Doe,Lawyer


In [14]:
leftjoin(people, jobs, on=:ID)

,ID,Name,Job
,Int64,String,String?
1,10,Timmy,missing
2,20,John Doe,Doctor
3,40,Jane Doe,Lawyer


In [15]:
rightjoin(people, jobs, on=:ID)

,ID,Name,Job
,Int64,String?,String
1,20,John Doe,Doctor
2,40,Jane Doe,Lawyer
3,60,missing,Astronaut


In [18]:
outerjoin(people, jobs, on=:ID)

,ID,Name,Job
,Int64,String?,String?
1,10,Timmy,missing
2,20,John Doe,Doctor
3,40,Jane Doe,Lawyer
4,60,missing,Astronaut


In [22]:
semijoin(people, jobs, on=:ID) 
#=
SELECT people.* 
FROM people 
LEFT JOIN jobs ON people.ID = jobs.ID
=#

,ID,Name
,Int64,String
1,20,John Doe
2,40,Jane Doe


In [24]:
antijoin(people, jobs, on=:ID)
#=
SELECT people.*
FROM people
LEFT JOIN jobs ON people.ID = jobs.ID
WHERE jobs.ID IS NULL
=#

,ID,Name
,Int64,String
1,10,Timmy


In [25]:
crossjoin(people, jobs, makeunique=true)

,ID,Name,ID_1,Job
,Int64,String,Int64,String
1,10,Timmy,20,Doctor
2,10,Timmy,40,Lawyer
3,10,Timmy,60,Astronaut
4,20,John Doe,20,Doctor
5,20,John Doe,40,Lawyer
6,20,John Doe,60,Astronaut
7,40,Jane Doe,20,Doctor
8,40,Jane Doe,40,Lawyer
9,40,Jane Doe,60,Astronaut


In [26]:
a = DataFrame(PID=[20, 40], Name=["John", "Jane"])
b = DataFrame(JID=[20, 40], Job=["Lawyer", "Doctor"])
innerjoin(a, b, on=:PID => :JID)

,PID,Name,Job
,Int64,String,String
1,20,John,Lawyer
2,40,Jane,Doctor


In [28]:
a = DataFrame(
    City=["Amsterdam", "Berlin", "Berlin", "Chicago", "Chicago"],
    Job=["Athlete", "Athlete", "Athlete", "Butler", "Butler"],
    Category=[1, 2, 3, 4,5])

,City,Job,Category
,String,String,Int64
1,Amsterdam,Athlete,1
2,Berlin,Athlete,2
3,Berlin,Athlete,3
4,Chicago,Butler,4
5,Chicago,Butler,5


In [30]:
b = DataFrame(
    Location=["Amsterdam", "Berlin", "Berlin", "Chicago", "Chicago"],
    Work=["Athlete", "Athlete", "Athlete", "Butler", "Butler"],
    Name=["a", "b", "c", "d", "e"])

,Location,Work,Name
,String,String,String
1,Amsterdam,Athlete,a
2,Berlin,Athlete,b
3,Berlin,Athlete,c
4,Chicago,Butler,d
5,Chicago,Butler,e


In [32]:
innerjoin(a, b, on=[Pair(:City, :Location), Pair(:Job, :Work)])

,City,Job,Category,Name
,String,String,Int64,String
1,Amsterdam,Athlete,1,a
2,Berlin,Athlete,2,b
3,Berlin,Athlete,2,c
4,Berlin,Athlete,3,b
5,Berlin,Athlete,3,c
6,Chicago,Butler,4,d
7,Chicago,Butler,4,e
8,Chicago,Butler,5,d
9,Chicago,Butler,5,e


In [33]:
innerjoin(
    a, b, on=[(:City, :Location), (:Job, :Work)], validate=(true, true))

ArgumentError: ArgumentError: Merge key(s) are not unique in both df1 and df2. First duplicate in df1 at 3. First duplicate in df2 at 3

In [34]:
a = DataFrame(ID=[20, 40], Name=["John", "Jane"])
b = DataFrame(ID=[20, 60], Job=["Lawyer", "Doctor"])

,ID,Job
,Int64,String
1,20,Lawyer
2,60,Doctor


In [35]:
outerjoin(a, b, on=:ID, validate=(true, true))

,ID,Name,Job
,Int64,String?,String?
1,20,John,Lawyer
2,40,Jane,missing
3,60,missing,Doctor


In [36]:
outerjoin(a, b, on=:ID, validate=(true, true), indicator=:source)

,ID,Name,Job,source
,Int64,String?,String?,Cat…
1,20,John,Lawyer,both
2,40,Jane,missing,left_only
3,60,missing,Doctor,right_only
